# 11.6 · 主题模型 / Topic Modeling

> **课程定位 / Where this fits**
> 第 6 课，**Part 11 · 经典 NLP**。
> Lesson 6, **Part 11 · Classic NLP**.
>
> 前几课都是**有监督**(有标签)。但现实中常有**一大堆没标签的文档**(新闻、评论、工单)，想自动发现"里面在讲哪些主题"。**主题模型(topic modeling)** 无监督地把语料分解成若干**主题**——每个主题是一组高频共现的词(如"太空主题"= orbit/launch/nasa…)，每篇文档是若干主题的混合。它用于文档聚类、内容推荐、探索性分析。本课用 **NMF** 和 **LDA** 在新闻语料上**自动发现主题**并可视化，看无监督能否"找回"我们已知的类别。
> Earlier lessons were **supervised** (labeled). But often we have **piles of unlabeled documents** (news, reviews, tickets) and want to auto-discover "what topics are inside." **Topic modeling** unsupervisedly decomposes a corpus into **topics** — each a set of co-occurring words (e.g. "space topic" = orbit/launch/nasa…), each document a mixture of topics. Used for clustering, recommendation, exploratory analysis. We use **NMF** and **LDA** to **auto-discover topics** on news data and visualize, checking whether unsupervised methods "recover" known categories.
>
> 💼 **实战/面试视角**："LDA vs NMF / 主题数怎么选 / 主题一致性 / 主题模型 vs 聚类" 是常考。
> 💼 **Practical/interview angle:** "LDA vs NMF / choosing #topics / coherence / topic models vs clustering" — common.

> 📐 **符号约定 / Notation**
> - 文档-词矩阵 $V$ —— 见 11.2 / document-term matrix
> - 主题 = 词的分布；文档 = 主题的分布 / topic = distribution over words; doc = distribution over topics

> 💡 **面试相关 / Interview-relevant**
> - "LDA 的生成过程/直觉"（出镜率 ★★★★）
> - "NMF 和 LDA 的区别"（★★★★，矩阵分解 vs 概率生成）
> - "怎么确定主题数 K / 评估主题质量"（★★★★，一致性 coherence）
> - "主题模型和聚类的区别"（★★★，软分配/混合）

---

## 学习目标 / Learning Objectives
1. 理解主题模型：文档=主题混合，主题=词分布。
   Understand topic models: doc = topic mixture, topic = word distribution.
2. 用 **NMF** 分解文档-词矩阵发现主题。
   Discover topics with NMF on the document-term matrix.
3. 用 **LDA** 做概率主题建模。
   Do probabilistic topic modeling with LDA.
4. 解读主题、看文档的主题混合、了解选 K 与评估。
   Interpret topics, inspect doc-topic mixtures, choosing K and evaluation.

## 目录 / TOC
1. [主题模型的直觉 ⭐](#1)
2. [NMF：矩阵分解发现主题 ⭐](#2)
3. [LDA：概率主题模型 ⭐](#3)
4. [解读、文档混合与选 K + 小结 ⭐](#4)


<a id="1"></a>
## 1. 主题模型的直觉 ⭐ / Intuition

核心假设：每篇文档**不是单一主题，而是若干主题的混合**(一篇文章可能 70% 讲太空、30% 讲技术)；每个主题是**一组经常一起出现的词**。
Core assumption: a document is **not one topic but a mixture** (70% space, 30% tech); each topic is **a set of words that co-occur**.

主题模型做的事：给定一堆文档(只有词，没有标签)，**自动找出 $K$ 个主题**(每个主题=哪些词、权重多少)，以及**每篇文档由这些主题怎么混合而成**。本质是把**文档-词矩阵**分解成两个小矩阵：**文档-主题** × **主题-词**。
What it does: given documents (words only, no labels), **find $K$ topics** (which words, with weights) and **how each document mixes them**. Essentially, factor the **document-term matrix** into **document-topic** × **topic-word**.

> **和聚类的区别(面试)**：聚类通常把每篇文档**硬分到一个簇**；主题模型给**软分配**(一篇文档是多个主题的混合比例)——更贴合"一篇文章可涉及多个话题"。
> **vs clustering (interview):** clustering usually **hard-assigns** each doc to one cluster; topic models give a **soft mixture** (proportions over topics) — fitting "an article can span multiple topics."

我们用 **20 Newsgroups** 的 4 个差异较大的类别，看主题模型能否无监督"找回"这些主题。
We use 4 distinct **20 Newsgroups** categories and check if topic models "recover" them unsupervised.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_20newsgroups
sns.set_theme(style="whitegrid")

cats = ["sci.space", "rec.sport.baseball", "comp.graphics", "talk.politics.mideast"]
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers","footers","quotes"))
print(f"{len(data.data)} 篇文档, 真实类别(主题模型看不到!): {[c.split('.')[-1] for c in data.target_names]}")
print("目标: 无监督地从词共现中发现 4 个主题, 看是否对应这 4 个真实类别")


<a id="2"></a>
## 2. NMF：矩阵分解发现主题 ⭐ / NMF: Topics via Matrix Factorization

**非负矩阵分解(NMF)**：把文档-词矩阵 $V$ (这里用 TF-IDF，$文档 \times 词$) 近似分解成两个**非负**矩阵相乘：
**Non-negative Matrix Factorization (NMF):** approximate the document-term matrix $V$ (TF-IDF here, $docs \times words$) as a product of two **non-negative** matrices:

$$V \approx W \times H, \quad W:(文档 \times 主题),\ H:(主题 \times 词)$$

- $H$ 的每一行 = 一个**主题**(在各词上的权重)→ 取权重最高的词就是该主题的"关键词"。
  Each row of $H$ = a **topic** (weights over words) → its top-weight words are the topic's keywords.
- $W$ 的每一行 = 一篇**文档的主题混合**(它由各主题以多大比例构成)。
  Each row of $W$ = a **document's topic mixture**.

"非负"很关键：权重只能加不能减，所以主题是**部件的叠加**(可解释)，不像 PCA 有负权重难解释(呼应 Part 6)。NMF 简单、快、主题常很清晰。
"Non-negativity" matters: weights only add, so topics are **additive parts** (interpretable), unlike PCA's hard-to-read negative weights (echoing Part 6). NMF is simple, fast, often gives crisp topics.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

K = 4                                                     # 主题数(故意设成和真实类别数一样) / #topics
tfidf = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.4)
V = tfidf.fit_transform(data.data)                        # 文档-词 TF-IDF 矩阵 / document-term matrix
terms = np.array(tfidf.get_feature_names_out())
print(f"文档-词矩阵: {V.shape}")

nmf = NMF(n_components=K, init="nndsvd", random_state=0, max_iter=400)
W = nmf.fit_transform(V)                                  # 文档-主题 / document-topic
H = nmf.components_                                       # 主题-词 / topic-word
def show_topics(H, terms, k=10):
    for ti, row in enumerate(H):
        top = row.argsort()[::-1][:k]                     # 该主题权重最高的词 / top words of this topic
        print(f"  主题 {ti}: {', '.join(terms[top])}")
print("NMF 发现的 4 个主题(每个主题的高权重词):")
show_topics(H, terms)
print("\n观察: 无监督发现的主题清晰对应 太空/棒球/图形/中东政治 → 词共现确实蕴含主题结构")


<a id="3"></a>
## 3. LDA：概率主题模型 ⭐ / LDA: Probabilistic Topic Model

**隐含狄利克雷分布(LDA)** 是更经典的**概率生成模型**。它假设文档是这样"生成"的(直觉)：
**Latent Dirichlet Allocation (LDA)** is the classic **probabilistic generative model**. It assumes a document is "generated" like this (intuition):
1. 先为这篇文档**掷骰子**决定它的**主题混合比例**(如 60% 太空、40% 技术)。
   Roll dice to pick the document's **topic proportions** (e.g. 60% space, 40% tech).
2. 要写每个词时：先按比例**选一个主题**，再从该主题的**词分布**里**抽一个词**。
   For each word: **pick a topic** by those proportions, then **draw a word** from that topic's word distribution.

LDA 做的是**反推**：观察到这些词，最可能的主题(词分布)和每篇文档的主题比例是什么？它输出和 NMF 类似的东西(主题=词分布，文档=主题分布)，但基于**概率**且每个主题是合法的概率分布。
LDA does the **inverse**: given the observed words, what topics (word distributions) and per-document proportions are most likely? It outputs similar things to NMF (topics=word distributions, docs=topic distributions) but **probabilistically**, with each topic a proper probability distribution.

> **LDA vs NMF(面试)**：LDA = 概率生成模型(有理论根基、给概率)；NMF = 线性代数矩阵分解(快、常更清晰)。两者实践都常用，LDA 用**词频(count)**、NMF 常用 **TF-IDF**。
> **LDA vs NMF (interview):** LDA = probabilistic generative model (principled, gives probabilities); NMF = linear-algebra factorization (fast, often crisper). Both common; LDA uses **counts**, NMF often **TF-IDF**.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

cv = CountVectorizer(stop_words="english", min_df=5, max_df=0.4)   # LDA 用词频 / LDA uses counts
Vc = cv.fit_transform(data.data); terms_c = np.array(cv.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=K, random_state=0, max_iter=20, learning_method="batch")
doc_topic = lda.fit_transform(Vc)                         # 文档-主题分布 / document-topic distribution
print("LDA 发现的 4 个主题:")
show_topics(lda.components_, terms_c)

# 可视化: 每个主题的 top 词条形图 / bar chart of top words per topic
fig, axes = plt.subplots(1, K, figsize=(15, 3.4))
for ti, ax in enumerate(axes):
    row = lda.components_[ti]; top = row.argsort()[::-1][:8]
    ax.barh(terms_c[top][::-1], (row[top]/row.sum())[::-1], color="#39c")
    ax.set_title(f"主题 {ti}", fontsize=10)
fig.suptitle("LDA 发现的主题(每主题的高概率词) — 无监督, 不知道真实类别"); plt.tight_layout(); plt.show()
print("LDA 也找回了 太空/棒球/图形/政治 这几个主题(词略有不同, 思想一致)")


<a id="4"></a>
## 4. 解读、文档混合与选 K + 小结 ⭐ / Interpret, Mixtures & Choosing K

主题模型给**软分配**：每篇文档是各主题的混合。我们看几篇文档的**主题分布**，并验证"主导主题"是否对应其真实类别。
Topic models give **soft assignments**: each doc is a topic mixture. Let's inspect a few documents' **topic distributions** and check whether the dominant topic matches the true category.


In [ ]:
# 文档-主题分布(LDA): 每行=一篇文档在4个主题上的占比 / doc-topic distribution
fig, ax = plt.subplots(figsize=(9, 4))
sample_idx = [np.where(data.target==c)[0][0] for c in range(K)]    # 每个真实类别取一篇 / one doc per true class
sns.heatmap(doc_topic[sample_idx], annot=True, fmt=".2f", cmap="Blues",
            yticklabels=[data.target_names[c].split('.')[-1] for c in range(K)],
            xticklabels=[f"主题{t}" for t in range(K)], ax=ax)
ax.set_xlabel("LDA 主题"); ax.set_ylabel("文档真实类别"); ax.set_title("文档-主题分布: 每篇文档主导一个主题(软分配)")
plt.tight_layout(); plt.show()

# 用"主导主题"和真实标签对齐, 看无监督主题与类别的吻合度 / align dominant topic with true label
from scipy.stats import mode
dom = doc_topic.argmax(1)                                 # 每篇文档的主导主题 / dominant topic
topic2cat = {t: mode(data.target[dom==t], keepdims=False).mode for t in range(K)}  # 主题→最常见真实类别 / map
mapped = np.array([topic2cat[t] for t in dom])
purity = (mapped == data.target).mean()
print(f"主题↔类别对齐纯度 = {purity:.3f}: 远高于随机(0.25), 无监督主题已较好吻合真实类别(没用任何标签!)")
print("(并非完美: LDA 有的主题会把'太空'和'政治'部分混在一起, 这正是无监督的固有难度)")
print("\n选主题数 K: 没有标签时靠 主题一致性(coherence)/困惑度(perplexity)/人工解读 来定")
print("评估: coherence 衡量主题内高频词是否真的语义相关; 也常结合下游任务效果")


```
主题模型: 无监督; 文档=主题混合, 主题=词分布; 分解 文档-词矩阵 → 文档-主题 × 主题-词
vs 聚类: 聚类硬分配一个簇; 主题模型软分配(混合比例), 适合一文多主题
NMF: 非负矩阵分解(V≈W×H), 用TF-IDF, 快/主题清晰/可解释(非负=部件叠加)
LDA: 概率生成模型(文档掷骰选主题→主题抽词), 用词频count, 有理论根基/给概率
解读: 每主题取高权重词=关键词; 每文档取主导主题; 可与已知类别对齐验证
选K/评估: 主题一致性coherence/困惑度perplexity/人工; K是关键超参
应用: 文档探索/聚类/推荐/趋势分析; 现代也有 BERTopic(嵌入+聚类)
```

### 💡 面试速查 / Interview cheat-sheet
1. **核心**: 文档=主题混合, 主题=词分布; 分解文档-词矩阵。
   Core: doc = topic mixture, topic = word distribution; factor the doc-term matrix.
2. **LDA vs NMF**: 概率生成模型(count) vs 矩阵分解(TF-IDF, 快/清晰)。
   LDA vs NMF: probabilistic generative (counts) vs factorization (TF-IDF, fast/crisp).
3. **vs 聚类**: 软分配(混合比例) vs 硬分配(一个簇)。
   vs clustering: soft mixture vs hard single-cluster.
4. **选 K/评估**: 一致性coherence/困惑度/人工解读。
   Choosing K/eval: coherence/perplexity/human inspection.
5. **可解释**: 每主题高权重词即关键词, 验证语义合理。
   Interpretable: a topic's top words are its keywords.

### 下一节 / Next
**11.7 命名实体识别(NER)**——从分类/主题转向**序列任务**：找出文本里的人名、地名、机构名等实体。我们会讲 BIO 标注体系、特征工程，理解 NER 作为序列标注问题的思路。
**11.7 Named Entity Recognition** — moving from classification/topics to **sequence tasks**: find entities (persons, locations, organizations) in text. We'll cover the BIO tagging scheme, features, and NER as a sequence-labeling problem.
